In [1]:
import timm
import torch
import fastai   

print(timm.__version__)
print(torch.__version__)
print(fastai.__version__)



1.0.12
2.5.1+cu124
2.7.18


## Load Model and Transforms

In [2]:
from PIL import Image
from urllib.request import urlopen


model_name = "hf_hub:timm/tf_mobilenetv3_large_100.in1k"
model = timm.create_model(model_name, pretrained=True, num_classes=0).eval()
model = model.to("cuda")

data_config = timm.data.resolve_model_data_config(model)
transforms = timm.data.create_transform(**data_config, is_training=False)



## Get Embeddings

In [3]:
img = Image.open(
    urlopen(
        "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/beignets-task-guide.png"
    )
)

with torch.inference_mode():
    output = model(transforms(img).unsqueeze(0).to("cuda"))

In [4]:
output.shape

torch.Size([1, 1280])

In [8]:
def get_batch_embeddings(image_dir, batch_size=32):
    """
    Get embeddings for all images in a directory using batch processing
    
    Args:
        image_dir (str): Path to directory containing images
        batch_size (int): Number of images to process at once
        
    Returns:
        torch.Tensor: Embeddings for all images (n_images x embedding_dim)
    """
    from pathlib import Path
    import torch
    from PIL import Image
    from tqdm.auto import tqdm
    
    image_files = []
    for ext in ['.jpg', '.jpeg', '.png', '.bmp']:
        image_files.extend(Path(image_dir).glob(f'*{ext}'))
    
    all_embeddings = []
    
    for i in tqdm(range(0, len(image_files), batch_size), desc="Processing images"):
        batch_files = image_files[i:i + batch_size]
        batch_images = []
        
        # Load and transform each image in the batch
        for img_path in batch_files:
            img = Image.open(img_path)
            img_tensor = transforms(img)
            batch_images.append(img_tensor)
            
        # Stack batch and get embeddings
        batch_tensor = torch.stack(batch_images)
        
        with torch.inference_mode():
            batch_embeddings = model(batch_tensor.to("cuda"))
            
        all_embeddings.append(batch_embeddings)
    
    # Concatenate all batches
    return torch.cat(all_embeddings, dim=0)


embeddings = get_batch_embeddings("../../data/", batch_size=1000)
embeddings.shape

Processing images:   0%|          | 0/17 [00:00<?, ?it/s]

torch.Size([16200, 1280])

In [9]:
import numpy as np

np.save("baseline_embeddings.npy", embeddings.cpu().numpy())

In [10]:
embeddings = np.load("baseline_embeddings.npy")
embeddings.shape

(16200, 1280)

In [11]:
embeddings.dtype

dtype('float32')